<a href="https://colab.research.google.com/github/engMohamedAbdAlslam/DRP_segmentation/blob/copilot%2Fdevelop-preprocessing-pipeline/notebooks/01_disease_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Disease (Lesion) Segmentation Preprocessing
**Dataset:** DDR (`ryanthao/fgadr`)
**Goal:** Download via KaggleHub, preprocess fundus images, save as `.npz`, visualize samples.
> Run on **Google Colab**. Set your Kaggle credentials before running.

## 1. Colab Repo Setup

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/DRP_segmentation')
if not repo_path.exists():
    !git clone https://github.com/engMohamedAbdAlslam/DRP_segmentation.git /content/DRP_segmentation

%cd /content/DRP_segmentation
!git checkout copilot/develop-preprocessing-pipeline
print('Repo ready at', Path.cwd())

## 2. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'kagglehub', 'opencv-python-headless', 'tqdm', 'matplotlib', 'scikit-learn', 'numpy', 'pandas'],
               check=True)
print('Dependencies ready.')

## 3. Kaggle Authentication

In [ ]:
from google.colab import userdata
import os

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(userdata.get('KAGGLE_TOKEN'))
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
print('Kaggle token ready!')

## 4. Repository Setup & Imports

In [ ]:
import sys
import shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib
import sklearn
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print('numpy', np.__version__)
print('opencv', cv2.__version__)
print('sklearn', sklearn.__version__)

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    raise FileNotFoundError('src directory not found. Make sure the repo setup cell ran successfully.')
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image, save_preprocessed

data_dir = repo_root / 'data'
raw_dir = data_dir / 'raw'
processed_dir = data_dir / 'processed'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print('Repo root:', repo_root)

## 5. Download DDR Dataset via KaggleHub

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_TOKEN')

import kagglehub

DATASET_SLUG = 'ryanthao/fgadr'
idrid_root = raw_dir / 'ddr'
idrid_root.mkdir(parents=True, exist_ok=True)

already_downloaded = any(idrid_root.rglob('*.jpg')) or any(idrid_root.rglob('*.png'))
if not already_downloaded:
    print(f'Downloading {DATASET_SLUG} ...')
    download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    if download_path != idrid_root:
        shutil.copytree(download_path, idrid_root, dirs_exist_ok=True)
    print('Download complete:', idrid_root)
else:
    print('Dataset already present at:', idrid_root)

all_files = list(idrid_root.rglob('*'))
print(f'Total files: {len(all_files)}')

## 6. Index Images & Masks

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

# DDR dataset has no lesion masks — images only
all_imgs = [p for p in idrid_root.rglob('*') if p.suffix.lower() in IMAGE_EXTS]

rows = [{'image_path': img, 'mask_paths': []} for img in sorted(all_imgs)]
df = pd.DataFrame(rows)
print(f'Total images: {len(df)} | With masks: {df["mask_paths"].apply(bool).sum()}')
df.head()

## 7. Train / Val / Test Split (70/15/15)

In [ ]:
if df.empty:
    raise RuntimeError('No images found — check dataset download path.')

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
splits = {'train': train_df, 'val': val_df, 'test': test_df}

## 8. Batch Preprocessing & Save

In [ ]:
config = PreprocessConfig(target_size=(512, 512), normalization='zero_one')
DATASET_NAME = 'idrid'
errors = []

for split_name, split_df in splits.items():
    out_dir = processed_dir / DATASET_NAME / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\nProcessing {split_name} ({len(split_df)} images)...')
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        try:
            img_path = row['image_path']
            mask_paths = row['mask_paths']
            combined_mask = None
            if mask_paths:
                img_arr = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img_arr is not None:
                    combined_mask = np.zeros(img_arr.shape[:2], dtype=np.uint8)
                    for mp in mask_paths:
                        m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
                        if m is not None:
                            combined_mask = np.maximum(combined_mask, m)
            result = preprocess_fundus_image(img_path, mask=combined_mask, config=config)
            out_file = out_dir / (img_path.stem + '.npz')
            save_preprocessed(result, out_file)
        except Exception as e:
            errors.append({'file': str(row['image_path']), 'error': str(e)})
            print(f'  WARNING: skipped {row["image_path"].name} — {e}')

print(f'\nDone. Errors: {len(errors)}')

## 9. Dataset Statistics

In [ ]:
print('=== Dataset Statistics ===')
for split_name, split_df in splits.items():
    print(f'{split_name}: {len(split_df)} images')

widths, heights = [], []
for _, row in df.iterrows():
    img = cv2.imread(str(row['image_path']))
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w); heights.append(h)

if widths:
    print(f'\nImage sizes (W x H):')
    print(f'  Width  — min:{min(widths)} max:{max(widths)} mean:{int(np.mean(widths))}')
    print(f'  Height — min:{min(heights)} max:{max(heights)} mean:{int(np.mean(heights))}')

## 10. Visualization — Sample Grid

In [ ]:
# DDR has no masks — show images only
samples = train_df.sample(min(4, len(train_df)), random_state=42)

fig, axes = plt.subplots(len(samples), 1, figsize=(6, 4 * len(samples)))
if len(samples) == 1:
    axes = [axes]

for i, (_, row) in enumerate(samples.iterrows()):
    img_bgr = cv2.imread(str(row['image_path']))
    if img_bgr is None:
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(f'Image: {row["image_path"].name}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('DDR — Sample Fundus Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save Processed Files to Google Drive
> Run this cell to persist the preprocessed `.npz` files across Colab sessions.
> Files will be saved to `My Drive/DRP_processed/` on your Google Drive.

In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

drive_dest = Path('/content/drive/MyDrive/DRP_processed')
drive_dest.mkdir(parents=True, exist_ok=True)

src = processed_dir / 'idrid'
if src.exists():
    shutil.copytree(str(src), str(drive_dest / 'idrid'), dirs_exist_ok=True)
    npz_count = len(list((drive_dest / 'idrid').rglob('*.npz')))
    print(f'Saved {npz_count} .npz files to Google Drive at: {drive_dest / "idrid"}')
else:
    print('No processed files found. Run Cell 8 first.')

## Next Steps
- Use preprocessed `.npz` files from `data/processed/idrid/` (or Google Drive) as input to a segmentation model
- Recommended architecture: **U-Net** or **DeepLabV3+** with a ResNet/EfficientNet encoder
- Metrics to track: **Dice coefficient**, **IoU**, **AUC-PR** per lesion type (EX, HE, MA, SE)
- Consider weighted loss (BCE + Dice) due to class imbalance in lesion masks